# Preprocessing of GLACIOCLIM point mass balance data (RGI 11, French Alps)

This notebook builds the French stake (point mass balance, PMB) dataset used by MassBalanceMachine, from the GLACIOCLIM observatory files.

**Pipeline**
1. (Optional) Unzip the raw GLACIOCLIM archives, then read all stake CSVs into a dictionary keyed by file name (`<glacier>_<period>_smb_<...>_<year>`).
2. Read the two glaciers stored in other formats: Sarennes (one Excel sheet per year) and Glacier Blanc (wide table of annual balances).
3. Convert Lambert II/III coordinates to WGS84 and bring all files into one DataFrame in the WGMS/MBM format.
4. Merge stakes closer than 10 m, drop NaN, zero and extreme values, and remove one faulty measurement.
5. Assign RGI IDs from the RGI v6 outlines (glacier names are replaced by the RGI names), then sample OGGM topography; drop stakes outside the outline or without topography.
6. Add the sky-view factor (SVF) from the per-glacier grids made by `1.4 Create_grids` (part 1).
7. Re-assign stake IDs by elevation and relabel some winter balances as annual.
8. Save `FR_wgms_dataset_all.csv` to `cfg.dataPath/WGMS/France/csv/`.

## Setting up:

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '../../../../')) # Add root of repo to import MBM

import pandas as pd
import os
import warnings
import massbalancemachine as mbm
import geopandas as gpd
import matplotlib.pyplot as plt
import glob
from cmcrameri import cm
from oggm import utils

# France-specific helpers (GLACIOCLIM file readers, Lambert conversion,
# check_period_consistency, flag_elevation_mismatch) and paths
# (path_PMB_GLACIOCLIM_raw, ...). The star import also provides tqdm.
from papers.Transformer_TF.PMB_sources.RGI_11_French_Alps.scripts.glacioclim_preprocess import *
from papers.Transformer_TF.PMB_sources.RGI_11_French_Alps.scripts.config_FR import *

# Helper functions shared across regions (OGGM, stake QC, SVF)
from papers.Transformer_TF.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids, merge_pmb_with_oggm_data
from papers.Transformer_TF.scripts.pmb_processing import rename_stakes_by_elevation, check_point_ids_contain_glacier, remove_close_points, check_multiple_rgi_ids, add_svf_from_rgi_zarr, plot_missing_svf_for_all_glaciers, add_svf_nearest_valid, plot_glacier_svf_with_points


warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Call config file (data paths, seed, etc.)
cfg = mbm.FranceConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

## Load data into a dictionary of DataFrames

Unzip the GLACIOCLIM files (only needed once; set `read_in = True`). `extract_glacioclim_files` empties the target folder first.

In [ ]:
# Set to True to (re-)extract the raw zip files into path_PMB_GLACIOCLIM_raw
read_in = False
if read_in == True:
    extract_glacioclim_files(cfg.dataPath + path_PMB_GLACIOCLIM_raw)

### Read all CSVs from "unzipped" into a single dictionary of DataFrames
The key is the file name, e.g. `stso_winter_smb_accu_2019`. The glacier and period are later parsed from it.

In [ ]:
stakes_csv_all = {}
for path, _, files in os.walk(cfg.dataPath + path_PMB_GLACIOCLIM_raw):
    # Find all .csv files in the current directory
    csv_files = glob.glob(os.path.join(path, '*.csv'))

    # Read each CSV file and add to dictionary
    for file in csv_files:
        try:
            # Use the filename as the key
            key = os.path.splitext(os.path.basename(file))[0]
            # Read the CSV file
            stakes_csv_all[key] = pd.read_csv(file)
        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

print(f"\nTotal number of files processed: {len(stakes_csv_all)}")
display(stakes_csv_all)

### Special case: Sarennes

The file does not have a regular CSV structure, so it cannot simply be read in. For this code to work, make a table in each sheet of `B_SARENNES_94-20.xls` with start_date, spring_date, end_date, X, Y, Z, winter mb, summer mb and annual mb (reading needs the `xlrd` package).

In [ ]:
sarennes_path = cfg.dataPath + 'GLACIOCLIM/point/Glacier de SARENNES_discontinued/B_SARENNES_94-20.xls'

# Read all sheets into a dictionary of dfs
all_sheets = pd.read_excel(sarennes_path, sheet_name=None)
sarennes_dfs = extract_sarennes_data(all_sheets)
display(sarennes_dfs)

### Special case: Glacier Blanc
Annual balances only, stored as a wide table (one column per stake). Dates are set to the hydrological year (1 October to 30 September).

In [ ]:
blanc_path = cfg.dataPath + 'GLACIOCLIM/point/Glacier BLANC_unusable/GlacierBlanc2000_2024_pointMB_mwe.xls'

# Skip the two title rows and the first (label) column
blanc_data = pd.read_excel(blanc_path, skiprows=2).iloc[:, 1:]

blanc_df = extract_blanc_data(blanc_data)

display(blanc_df)

## Convert from Lambert III / Lambert II coordinates to WGS84
Mont-Blanc (`mdg_*`) and Argentière files are in Lambert II; all others, including three Saint-Sorlin files with columns misnamed `*_lambert2e`, are in Lambert III.

This cell changes `stakes_csv_all` in place, so it cannot be run twice: re-run from "Read all CSVs" first.

In [ ]:
## For some reason there is a 2 in front of the y_lambert2e coordinates in certain years, hardcode remove them:
stakes_csv_all['mdg_Tacul_winter_smb_abl_2007'][
    'y_lambert2e'] = stakes_csv_all['mdg_Tacul_winter_smb_abl_2007'][
        'y_lambert2e'].apply(lambda x: x - 2000000 if x > 2000000 else x)
stakes_csv_all['mdg_Talefre_annual_smb_abl_2006'][
    'y_lambert2e'] = stakes_csv_all['mdg_Talefre_annual_smb_abl_2006'][
        'y_lambert2e'].apply(lambda x: x - 2000000 if x > 2000000 else x)

# lambert_transform picks Lambert II or III from the file name
# (see the note in the markdown above); lamberttoWGS84 adds lat/lon and
# drops the x/y columns in place
stakes_csv_all = lambert_transform(stakes_csv_all)
sarennes_dfs = lambert_transform(sarennes_dfs)
blanc_df = lamberttoWGS84(blanc_df)  # Lambert III

## WGMS mapping, appending Sarennes, and conversion into a single DataFrame
Each GLACIOCLIM file is converted with `transform_WGMS_df`: the glacier and period come from the file name, and `POINT_ID` is `<file>_<profile>_setup<year>_<stake>`.

In [ ]:
stakes_csv_WGMS = {}

# Process each DataFrame in the original dictionary to WGMS format
# (files without the required columns are skipped silently)
for key, df in stakes_csv_all.items():
    try:
        required_cols = {
            'lat', 'lon', 'altitude', 'stake_number', 'day_start',
            'month_start', 'year_start', 'day_end', 'month_end', 'year_end'
        }
        if all(col in df.columns for col in required_cols):
            stakes_csv_WGMS[key] = transform_WGMS_df(df, key)
    except Exception as e:
        print(f"Error processing {key}: {str(e)}")

# Append SARENNES to the dictionary
columns = [
    'POINT_ID', 'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE',
    'TO_DATE', 'POINT_BALANCE', 'GLACIER', 'PERIOD', 'GLACIER_ZONE'
]
for key, df in sarennes_dfs.items():
    stakes_csv_WGMS[key] = df.rename(columns={
        'lat': 'POINT_LAT',
        'lon': 'POINT_LON'
    })[columns]

# Rename Glacier Blanc
blanc_df = blanc_df.rename(columns={'lat': 'POINT_LAT', 'lon': 'POINT_LON'})

Combine the dictionary of DataFrames into one DataFrame:

In [ ]:
# Convert dictionary to single DataFrame
df_pmb_all_raw = pd.concat(stakes_csv_WGMS.values(), ignore_index=True)
# Append Glacier Blanc
df_pmb_all_raw = pd.concat([df_pmb_all_raw, blanc_df], ignore_index=True)
# Sort by date and glacier
df_pmb_all_raw = df_pmb_all_raw.sort_values(
    ['GLACIER', 'GLACIER_ZONE', 'PERIOD', 'FROM_DATE'])
# Add YEAR column to df (year of TO_DATE = hydrological year)
df_pmb_all_raw['YEAR'] = df_pmb_all_raw['TO_DATE'].astype(str).str[:4].astype(
    int)
# Add data modification column to keep track of mannual changes
df_pmb_all_raw['DATA_MODIFICATION'] = ''
# Add remark that glacier blanc dates were modified
df_pmb_all_raw.loc[df_pmb_all_raw['GLACIER'] == 'glacier_blanc',
                   'DATA_MODIFICATION'] = 'Hydraulic year dates'
df_pmb_all_raw.head(3)

## General data cleaning
### Merge stakes that are close
Within each glacier, year and period, stakes closer than 10 m are merged into one point with the mean `POINT_BALANCE` (see `remove_close_points`). Summer balances (and WGMS `index` balances) are removed first.

In [ ]:
# Keep only annual and winter balances (drops summer and WGMS 'index' rows)
df_pmb_all_raw = df_pmb_all_raw[df_pmb_all_raw['PERIOD'].isin(['annual', 'winter'])]

# Merge stakes < 10 m apart, glacier by glacier
df_pmb_all_raw_dropped = pd.DataFrame()
for gl in tqdm(df_pmb_all_raw.GLACIER.unique(), desc='Merging stakes'):
    print(f'-- {gl.capitalize()}:')
    df_gl = df_pmb_all_raw[df_pmb_all_raw.GLACIER == gl]
    df_gl_cleaned = remove_close_points(df_gl)
    df_pmb_all_raw_dropped = pd.concat([df_pmb_all_raw_dropped, df_gl_cleaned])
# Drop temporary projected coordinates added by remove_close_points
df_pmb_all_raw_dropped.drop(['x', 'y'], axis=1, inplace=True)
df_pmb_all_raw_dropped.reset_index(inplace=True, drop=True)

### Cleaning
The first (commented-out) cell is an older version of the second one.

In [ ]:
# Older version of the next cell, kept for reference (not run)
# print("Rows with NaN values:")
# display(df_pmb_all_raw_dropped[df_pmb_all_raw_dropped.isna().any(axis=1)])

# print("Rows with zero values:")
# display(df_pmb_all_raw_dropped[df_pmb_all_raw_dropped.eq(0).any(axis=1)])

# print("Rows with extreme POINT_BALANCE values (>5 or <-15):")
# display(
#     df_pmb_all_raw_dropped[(df_pmb_all_raw_dropped['POINT_BALANCE'] > 5) |
#                            (df_pmb_all_raw_dropped['POINT_BALANCE'] < -15)])

# # Saint-Sorlin has POINT_ELEVATION 0.0 on 8 stakes, also about 20 stakes have point balance 0.00, remove them all
# mask_zeros = df_pmb_all_raw_dropped.eq(0).any(axis=1)
# df_pmb_all_raw_dropped = df_pmb_all_raw_dropped[~mask_zeros]

# # stso_summer_smb_abl_2017_ablation_setup2015_14 -88 pmb, nonsensical value, remove it
# mask = df_pmb_all_raw_dropped[
#     'POINT_ID'] != 'stso_summer_smb_abl_2017_ablation_setup2015_14'
# df_pmb_all_raw_dropped = df_pmb_all_raw_dropped[mask]

In [ ]:
# --- counts before ---
n0 = len(df_pmb_all_raw_dropped)

# Drop any rows containing NaN
mask_nan = df_pmb_all_raw_dropped.isna().any(axis=1)
print(f"Dropping {mask_nan.sum()} rows with at least one NaN (out of {n0}).")
df_pmb_all_raw_dropped = df_pmb_all_raw_dropped.loc[~mask_nan].copy()

# Drop rows with zeros in elevation or balance
# (e.g. Saint-Sorlin has 8 stakes with elevation 0 and ~20 with balance 0.00)
zero_cols = ["POINT_ELEVATION", "POINT_BALANCE"]  # add others if needed
mask_zeros = (df_pmb_all_raw_dropped[zero_cols] == 0).any(axis=1)
print(f"Dropping {mask_zeros.sum()} rows with zero in {zero_cols}.")
df_pmb_all_raw_dropped = df_pmb_all_raw_dropped.loc[~mask_zeros].copy()

# Drop extreme balances
mask_extreme = (df_pmb_all_raw_dropped["POINT_BALANCE"]
                > 5) | (df_pmb_all_raw_dropped["POINT_BALANCE"] < -15)
print(
    f"Dropping {mask_extreme.sum()} rows with extreme POINT_BALANCE (>5 or <-15)."
)
df_pmb_all_raw_dropped = df_pmb_all_raw_dropped.loc[~mask_extreme].copy()

# Drop specific bad point id (summer balance of -88 m w.e.)
bad_id = "stso_summer_smb_abl_2017_ablation_setup2015_14"
mask_bad = df_pmb_all_raw_dropped["POINT_ID"] == bad_id
print(f"Dropping {mask_bad.sum()} rows with POINT_ID == {bad_id}.")
df_pmb_all_raw_dropped = df_pmb_all_raw_dropped.loc[~mask_bad].copy()

print(
    f"Total dropped: {n0 - len(df_pmb_all_raw_dropped)} rows. Remaining: {len(df_pmb_all_raw_dropped)}."
)

In [ ]:
# Flags annual periods outside 9-15 months, winter outside 4-9, summer outside 3-8
annual_inconsistent, winter_inconsistent, summer_inconsistent = check_period_consistency(
    df_pmb_all_raw_dropped)

display(annual_inconsistent)
display(summer_inconsistent)

# 7 short summers but nothing majorly inconsistent, leaving them in
# Annuual, mdg_langue_annual_smb_abl_2008_setup2008_9 is a faulty measurement, goes from 2009 to 2008, pop it
mask = df_pmb_all_raw_dropped[
    'POINT_ID'] != 'mdg_langue_annual_smb_abl_2008_Langue_setup2008_9'
df_pmb_all_raw_dropped = df_pmb_all_raw_dropped[mask]
df_pmb_all_raw_dropped.reset_index(inplace=True, drop=True)

In [ ]:
# Number of measurements per year (top) and per glacier (bottom)
print('Number of winter and annual samples:', len(df_pmb_all_raw_dropped))
print('Number of annual samples:',
      len(df_pmb_all_raw_dropped[df_pmb_all_raw_dropped.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_all_raw_dropped[df_pmb_all_raw_dropped.PERIOD == 'winter']))

# Number of measurements per year:
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_all_raw_dropped.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar',
    stacked=True,
    color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

ax = axs.flatten()[1]
num_gl = df_pmb_all_raw_dropped.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()

#### Add RGI ID and OGGM data
RGI IDs are assigned by intersecting the stakes with the RGI v6 outlines of region 11, and the GLACIOCLIM glacier names are replaced by the RGI names.

In [ ]:
# Load glacier outlines
rgi_file = utils.get_rgi_region_file(region="11", version="62")
glacier_outline = gpd.read_file(rgi_file)

# Add RGI IDs through intersection
df_pmb = mbm.data_processing.utils.get_rgi(data=df_pmb_all_raw_dropped,
                                           glacier_outlines=glacier_outline)

print('Number of measurements without RGI:',
      len(df_pmb[df_pmb['RGIId'].isna()]))

# Remove stakes without RGIId, as they wont have OGGM data anyways
print('Drop stakes without RGIs...')
df_pmb = df_pmb.dropna(subset=['RGIId'])

# Create a dictionary mapping from RGIId to new glacier names
rgi_to_name_dict = dict(zip(glacier_outline.RGIId, glacier_outline.Name))
df_pmb['GLACIER'] = df_pmb['RGIId'].map(rgi_to_name_dict)

# Check that each glacier name maps to a single RGI ID
rgiids6 = df_pmb[['GLACIER', 'RGIId']].drop_duplicates()
if check_multiple_rgi_ids(rgiids6):
    print(
        "-- Alert: The following glaciers have more than one RGIId. Cleaning up."
    )
    # Different glaciers share the same RGI name: make the names unique
    # by appending the RGI ID, e.g. "<name>_RGI60-11.xxxxx"
    n_rgi = rgiids6.groupby('GLACIER')['RGIId'].nunique()
    dup_names = n_rgi[n_rgi > 1].index
    display(rgiids6[rgiids6['GLACIER'].isin(dup_names)])
    mask_dup = df_pmb['GLACIER'].isin(dup_names)
    df_pmb.loc[mask_dup, 'GLACIER'] = (df_pmb.loc[mask_dup, 'GLACIER'] + '_' +
                                       df_pmb.loc[mask_dup, 'RGIId'])
    assert not check_multiple_rgi_ids(df_pmb[['GLACIER', 'RGIId']].drop_duplicates())
    df_pmb_clean = df_pmb
else:
    print("-- All glaciers are correctly associated with a single RGIId.")
    df_pmb_clean = df_pmb

df_pmb_clean.reset_index(inplace=True, drop=True)

In [ ]:
# Add OGGM data:
# initialize OGGM glacier directories (only the stake glaciers)
gdirs, rgi_ids_list = initialize_oggm_glacier_directories(
    cfg,
    rgi_region="11",
    rgi_version="62",
    base_url=
    "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
    log_level='WARNING',
    task_list=None,
    rgi_ids_list=df_pmb_clean.RGIId.unique().tolist())

# Save the OGGM gridded data of these glaciers to disk
export_oggm_grids(cfg,
                  gdirs,
                  rgi_region="11",
                  subset_rgis=df_pmb_clean.RGIId.unique().tolist())

# Sample topographic variables at each stake and flag stakes within the glacier outline
# (default variables: aspect, slope, topo, hugonnet_dhdt, consensus_ice_thickness, millan_v)
df_pmb_topo = merge_pmb_with_oggm_data(
    df_pmb=df_pmb_clean,
    gdirs=gdirs,
    rgi_region="11",
    rgi_version="62",
    verbose=True,
)
# Restrict to within glacier shape and drop the column
df_pmb_topo = df_pmb_topo[df_pmb_topo['within_glacier_shape'] == True]
df_pmb_topo = df_pmb_topo.drop(columns=['within_glacier_shape'])

# Drop points if aspect, slope or topo is NaN
df_pmb_topo = df_pmb_topo[~df_pmb_topo.aspect.isna()]
df_pmb_topo = df_pmb_topo[~df_pmb_topo.slope.isna()]
df_pmb_topo = df_pmb_topo[~df_pmb_topo.topo.isna()]

df_pmb_topo.reset_index(inplace=True, drop=True)
df_pmb_topo.head()

In [ ]:
# Diagnostic: DEM source and grid resolution of the OGGM glacier directories
from oggm import utils
stats = utils.compile_glacier_statistics(gdirs)
print(stats['dem_source'].value_counts())

# grid resolution per glacier
print(pd.Series([gd.grid.dx for gd in gdirs]).describe())

In [ ]:
print('Number of winter and annual samples:', len(df_pmb_topo))
print('Number of annual samples:',
      len(df_pmb_topo[df_pmb_topo.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_topo[df_pmb_topo.PERIOD == 'winter']))
# Unique glaciers, sorted
glacier_list = sorted(df_pmb_topo.GLACIER.unique())
print(f"Number of glaciers: {len(glacier_list)}")
print(f"Glaciers: {glacier_list}")

In [ ]:
# Number of measurements per year (top) and per glacier (bottom)
# Number of measurements per year:
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_topo.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar',
    stacked=True,
    color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

ax = axs.flatten()[1]
num_gl = df_pmb_topo.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()

## Check for wrong elevations:
Lists stakes whose reported elevation differs from the OGGM DEM (`topo`) by more than 400 m. This is a diagnostic only: the flagged rows are not removed.

In [ ]:
# Diagnostic only: df_checked / df_bad are not used further
df_checked, df_bad = flag_elevation_mismatch(df_pmb_topo, threshold=400)

## Add sky-view factor:
The SVF is sampled at each stake from the per-glacier grids (`<RGIId>.zarr` in `xr_masked_grids/`). Stakes on a masked (NaN) pixel are first plotted, then filled with the nearest valid SVF pixel.

In [ ]:
# Directory with the per-glacier SVF grids (<RGIId>.zarr)
path_masked_xr = os.path.join(cfg.dataPath,
                              "RGI_v6/RGI_11_CentralEurope/xr_masked_grids")

# Nearest-pixel SVF sampling (NaN if the pixel is masked)
df_pmb_topo_svf = add_svf_from_rgi_zarr(
    df_pmb_topo,
    path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_var="svf",
    out_col="svf",
)
# Stakes that fell on a masked (NaN) SVF pixel
df_missing = df_pmb_topo_svf[df_pmb_topo_svf["svf"].isna()].copy()
print("Missing SVF points:", len(df_missing))
print("Glaciers affected:", sorted(df_missing["RGIId"].unique()))

In [ ]:
# Plot SVF grids with missing-SVF stakes (red) and valid stakes (white) per glacier
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

In [ ]:
# Sample SVF again; points on masked pixels are filled with the nearest
# valid pixel within max_radius grid cells. This is the SVF that is kept.
df_pmb_topo_svf_new = add_svf_nearest_valid(
    df_pmb_topo,
    path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_var="svf",
    out_col="svf",
    max_radius=30,  # ~30 grid cells search; adjust if needed
)

print("Missing SVF points after nearest-valid fill:",
      df_pmb_topo_svf_new["svf"].isna().sum())

# Same plot after filling: there should be no red points left
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf_new,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

In [ ]:
# One plot per glacier: SVF grid with the stakes coloured by their sampled SVF
plot_glacier_svf_with_points(
    df_with_svf=df_pmb_topo_svf_new,
    path_masked_xr=path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_col="svf",
    svf_var="svf",
    save_dir=
    None,  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_point_plots")
)

## Give new stake IDs:
Stake IDs are re-assigned per glacier as `<GLACIER>_<rank>`, where the rank is the stake's mean elevation (0 = lowest).

In [ ]:
# Rename POINT_IDs to <GLACIER>_<rank>, ranked by elevation
df_pmb_new_ids = rename_stakes_by_elevation(df_pmb_topo_svf_new)

# Check that every POINT_ID contains its glacier name
check_point_ids_contain_glacier(df_pmb_new_ids)

print('Number of winter and annual samples:', len(df_pmb_new_ids))
print('Number of annual samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'winter']))

# Histogram of mass balance
df_pmb_new_ids['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Final cleaning:
"Winter" measurements that end in June with a negative balance are relabelled as annual. Unlike the other regions, no measurements are removed based on their months.

In [ ]:
df_pmb_clean = df_pmb_new_ids.copy()
# Months (MM) of start and end of each measurement
df_pmb_clean['MONTH_START'] = [
    str(date)[4:6] for date in df_pmb_clean.FROM_DATE
]
df_pmb_clean['MONTH_END'] = [str(date)[4:6] for date in df_pmb_clean.TO_DATE]

winter_pmb = df_pmb_clean[df_pmb_clean.PERIOD == "winter"]
annual_pmb = df_pmb_clean[df_pmb_clean.PERIOD == "annual"]

print("Winter measurement months:")
print("  Unique start months:", sorted(winter_pmb["MONTH_START"].unique()))
print("  Unique end months:  ", sorted(winter_pmb["MONTH_END"].unique()))

print("\nAnnual measurement months:")
print("  Unique start months:", sorted(annual_pmb["MONTH_START"].unique()))
print("  Unique end months:  ", sorted(annual_pmb["MONTH_END"].unique()))

# Correct mislabeled winter MB:
# If MONTH_END == 6 and MB is negative, it should be annual (not winter):
# the balance already includes melt
mask_fix = ((df_pmb_clean["PERIOD"] == "winter") &
            (df_pmb_clean["MONTH_END"] == '06') &
            (df_pmb_clean["POINT_BALANCE"] < 0))

print("Rows to relabel winter -> annual:", mask_fix.sum())

df_pmb_clean.loc[mask_fix, "PERIOD"] = "annual"

# For now, don't remove anything:
# Save to csv:
df_pmb_clean.to_csv(os.path.join(cfg.dataPath,
                                 'WGMS/France/csv/FR_wgms_dataset_all.csv'),
                    index=False)

# Histogram of mass balance
df_pmb_clean['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Plot stakes and glaciers:

In [ ]:
# Interactive map of all cleaned stakes, one layer per glacier
from papers.Transformer_TF.scripts.plotting.folium import plot_stakes_folium

m = plot_stakes_folium(df_pmb_clean, color_map=None, layer_control=True)
m